In [1]:
from environment import Environment
from nnsight import LanguageModel
import sys
import torch as t
from getpass import getpass
import os

# REPLICATE_API_TOKEN = getpass()
# os.environ["REPLICATE_API_TOKEN"] = "<YOUR KEY HERE>"

Using OpenAI API


In [2]:
model = LanguageModel("openai-community/gpt2", device_map='auto', dispatch=True)

In [3]:
# !git clone https://github.com/jbloomAus/mats_sae_training.git
# %cd mats_sae_training
# !pip install -r requirements.txt

sys.path.append("../mats_sae_training")

from sae_training.sparse_autoencoder import SparseAutoencoder

t.set_grad_enabled(False)

In [4]:
from huggingface_hub import hf_hub_download

REPO_ID = "jbloom/GPT2-Small-SAEs"

sae_list = []
n_layers = 12

local_dir = "../jbloom_dictionaries"

for layer in range(n_layers):
    filename =  f"final_sparse_autoencoder_gpt2-small_blocks.{layer}.hook_resid_pre_24576.pt"
    resid_sae = hf_hub_download(repo_id = REPO_ID, filename = filename, local_dir=local_dir)

    save_path = f"{local_dir}/{filename}"
    sae = SparseAutoencoder.load_from_pretrained(save_path)
    sae.to("cuda:0")
    
    sae_list.append(sae)

print("Loaded dictionaries")

Loaded dictionaries


In [5]:
env = Environment(
    model=model, 
    sae_list=sae_list,
)

In [6]:
env.load(
    layer=10,
    feature_id=7000,
)

100%|██████████| 13/13 [00:37<00:00,  2.87s/it]

Activation Cache Size: torch.Size([1950, 128, 24576])


In [7]:
long_explanation_list = env.explainer.explain()

env.state.history.append({"role":"system","message":long_explanation_list})

Processing batches: 100%|██████████| 2/2 [00:28<00:00, 14.36s/it]


In [9]:
from condenser import condense
from prompting import get_simple_condenser_template

explanation_list, output = condense(long_explanation_list, return_output=True)

env.state.history.append({"role":"section","message":"Running condenser."})
env.state.history.append({"role":"user","message":get_simple_condenser_template(long_explanation_list)})
env.state.history.append({"role":"assistant","message":"".join(output)})

In [10]:
d_scores_list = env.d_scorer.score(explanation_list, l_ctx=15, r_ctx=4)
pruned_explanation_list = [explanation_list[i] for i in range(len(explanation_list)) if d_scores_list[i][0] > 0.6 and d_scores_list[i][1] < 0.3]

pruned_explanation_list

100%|██████████| 2/2 [00:43<00:00, 21.91s/it]


['The neuron activates on forms of the verb "to be" or its contractions, used in statements of certainty or description. The neuron also activates on forms of the verb "to be" or its contractions when followed by verbs or adjectives describing actions or states.',
 'First-person pronouns followed by contractions of the verb "to be", indicating personal state or action. Use of "I" or "we" followed by a form of "to be", typically introducing or continuing a description of the speaker\'s thoughts or actions.']

In [11]:
g_scores_list = env.gen_scorer.score(explanation_list, sae_list[layer])

IndexError: list index out of range

In [17]:
import importlib
import utils
importlib.reload(utils)

utils.log_conversation(env.state.history, "oai_conversation.txt")

Using OpenAI API


┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Role      ┃ Content                                                                                                  ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Section   │ Running explainer.                                                                                       │
├───────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ User      │                                                                                                          │
│           │     <|begin_of_text|>                                                                                    │
│           │                                                                                                          │
│           │     <|start_header_id|>system<|end_header_id|>                                                           │
│           │                                                                                                          │
│           │                                                                                                          │
│           │  You are a meticulous AI researcher conducting an important investigation into a certain neuron in a     │
│           │ language model.                                                                                          │
│           │                                                                                                          │
│           │ You will be given a list of text examples on which the neuron activates. The specific tokens which cause │
│           │ the neuron to activate will appear between delimiters like <<this>>. If a sequence of consecutive tokens │
│           │ all cause the neuron to activate, the entire sequence of tokens will be contained between delimiters     │
│           │ <<just like this>>.                                                                                      │
│           │                                                                                                          │
│           │ Your task is to understand what features of the input text cause the neuron to activate.                 │
│           │                                                                                                          │
│           │ You must follow these steps:                                                                             │
│           │                                                                                                          │
│           │ Step 1: For each text example in turn, note which tokens (i.e. words, fragments of words, or symbols)    │
│           │ caused the neuron to activate. Then note which tokens came before the activating tokens. Then note which │
│           │ tokens came after the activating tokens.                                                                 │
│           │ Step 2: Look for patterns in the tokens you noted down in Step 1.                                        │
│           │ Step 3: Write down several general shared features of the text examples.                                 │
│           │ Step 4: Look at the shared features you found, as well as patterns in the tokens you wrote down in Steps │
│           │ 1 and 2, to produce 3 explanations for what features of text cause the neuron to activate. The final 2   │
│           │ lines of your output must consist of your 2 explanations.                                                │
│           │                                                                                                          │
│           │ Guidelines:                                                                                              │
│      

In [ ]:
for i in range(len(explanation_list)):
  print(f"Explanation: {explanation_list[i]}")
  print(f"Detection rate: {d_scores_list[i][0]:.2f}")
  print(f"False positive rate: {d_scores_list[i][1]:.2f}")
  print(f"Generation score: {g_scores_list[i]:.2f}")
  print()